# Model initialization

In [1]:
%load_ext autoreload
%autoreload 2

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import pandas as pd
import numpy as np
from PIL import Image
import os
import nibabel as nib

from model import CTScreener

model = CTScreener()

INFO 02-23 16:49:32 [utils.py:253] non-default args: {'trust_remote_code': True, 'allowed_local_media_path': '/home/a-lugovoi/Git/medscreen/dev/', 'dtype': 'bfloat16', 'seed': None, 'max_model_len': 8192, 'gpu_memory_utilization': 0.3, 'disable_log_stats': True, 'model': 'google/medgemma-1.5-4b-it'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


WARNING 02-23 16:49:32 [arg_utils.py:1175] `seed=None` is equivalent to `seed=0` in V1 Engine. You will no longer be allowed to pass `None` in v0.13.
INFO 02-23 16:49:33 [model.py:637] Resolved architecture: Gemma3ForConditionalGeneration
INFO 02-23 16:49:33 [model.py:1750] Using max model len 8192
INFO 02-23 16:49:34 [scheduler.py:228] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=297759) INFO 02-23 16:49:40 [core.py:93] Initializing a V1 LLM engine (v0.12.0) with config: model='google/medgemma-1.5-4b-it', speculative_config=None, tokenizer='google/medgemma-1.5-4b-it', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_ou

(EngineCore_DP0 pid=297759) Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


(EngineCore_DP0 pid=297759) INFO 02-23 16:49:58 [gpu_model_runner.py:3467] Starting to load model google/medgemma-1.5-4b-it...
(EngineCore_DP0 pid=297759) INFO 02-23 16:49:58 [layer.py:500] Using AttentionBackendEnum.FLASH_ATTN for MultiHeadAttention in multimodal encoder.
(EngineCore_DP0 pid=297759) INFO 02-23 16:49:58 [cuda.py:411] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION']


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore_DP0 pid=297759) INFO 02-23 16:50:02 [default_loader.py:308] Loading weights took 2.88 seconds
(EngineCore_DP0 pid=297759) INFO 02-23 16:50:03 [gpu_model_runner.py:3549] Model loading took 8.5834 GiB memory and 3.962426 seconds
(EngineCore_DP0 pid=297759) INFO 02-23 16:50:03 [gpu_model_runner.py:4306] Encoder cache will be initialized with a budget of 8192 tokens, and profiled with 31 image items of the maximum feature size.
(EngineCore_DP0 pid=297759) INFO 02-23 16:50:13 [backends.py:655] Using cache directory: /home/a-lugovoi/.cache/vllm/torch_compile_cache/00f8fdbe2b/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=297759) INFO 02-23 16:50:13 [backends.py:715] Dynamo bytecode transform time: 7.98 s
(EngineCore_DP0 pid=297759) INFO 02-23 16:50:18 [backends.py:216] Directly load the compiled graph(s) for dynamic shape from the cache, took 3.664 s
(EngineCore_DP0 pid=297759) INFO 02-23 16:50:19 [monitor.py:34] torch.compile takes 11.65 s in total
(EngineCore_DP

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|███████████████████████████████| 51/51 [00:02<00:00, 18.98it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████████████████████████████████████████████| 35/35 [00:01<00:00, 24.12it/s]


(EngineCore_DP0 pid=297759) INFO 02-23 16:50:27 [gpu_model_runner.py:4466] Graph capturing finished in 5 secs, took 2.27 GiB
(EngineCore_DP0 pid=297759) INFO 02-23 16:50:27 [core.py:254] init engine (profile, create kv cache, warmup model) took 23.99 seconds
INFO 02-23 16:50:30 [llm.py:343] Supported tasks: ['generate']


# MosMedData: Chest CT Scans with Signs of COVID-19
- https://mosmed.ai/en/datasets/datasets/covid191110/

In [2]:
DATA_DIR = "data/MosMedData_COVID19"
GROUND_TRUTH = {"CT-0": "NORMAL", "CT-1": "ABNORMAL", "CT-2": "ABNORMAL", "CT-3": "ABNORMAL", "CT-4": "ABNORMAL"}

rows = []
for group in sorted(GROUND_TRUTH):
    group_dir = os.path.join(DATA_DIR, group)
    files = sorted([f for f in os.listdir(group_dir) if f.endswith(".nii.gz")])
    print(f"\n{group}: {len(files)} studies (label={GROUND_TRUTH[group]})")

    for i, fname in enumerate(files):
        nii = nib.load(os.path.join(group_dir, fname))
        volume = nii.get_fdata().astype(np.float32)
        volume = np.transpose(volume, (2, 1, 0))

        result = model.run(volume)
        rows.append({
            "group": group,
            "file": fname,
            "label": GROUND_TRUTH[group],
            "verdict": result["verdict"],
            "abnormal_ratio": result["abnormal_ratio"],
            "n_slices": result["total_slices"],
            "time": round(result["inference_time"], 1),
        })
        print(f"  [{i+1}/{len(files)}] {fname}: {result['verdict']} ({result['abnormal_ratio']:.0%}) — {result['inference_time']:.1f}s")

df = pd.DataFrame(rows)
df.to_csv("eval_mosmed.csv", index=False)
print(f"\nDone! {len(df)} studies processed.")


CT-0: 100 studies (label=NORMAL)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


INFO 02-23 17:07:22 [chat_utils.py:574] Detected the chat template content format to be 'openai'. You can set `--chat-template-content-format` to override this.
  [1/100] study_0001.nii.gz: NORMAL (0%) — 35.8s
  [2/100] study_0002.nii.gz: NORMAL (0%) — 5.6s
  [3/100] study_0003.nii.gz: ABNORMAL (33%) — 5.6s
  [4/100] study_0004.nii.gz: ABNORMAL (33%) — 5.7s
  [5/100] study_0005.nii.gz: NORMAL (0%) — 5.6s
  [6/100] study_0006.nii.gz: NORMAL (0%) — 4.8s
  [7/100] study_0007.nii.gz: ABNORMAL (100%) — 5.6s
  [8/100] study_0008.nii.gz: NORMAL (0%) — 4.7s
  [9/100] study_0009.nii.gz: NORMAL (0%) — 4.8s
  [10/100] study_0010.nii.gz: NORMAL (0%) — 6.1s
  [11/100] study_0011.nii.gz: NORMAL (0%) — 5.6s
  [12/100] study_0012.nii.gz: NORMAL (0%) — 5.6s
  [13/100] study_0013.nii.gz: NORMAL (0%) — 5.5s
  [14/100] study_0014.nii.gz: ABNORMAL (33%) — 5.8s
  [15/100] study_0015.nii.gz: NORMAL (0%) — 5.8s
  [16/100] study_0016.nii.gz: NORMAL (0%) — 9.0s
  [17/100] study_0017.nii.gz: NORMAL (0%) — 5.7s
 

## Results

In [3]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

y_true = (df["label"] == "ABNORMAL").astype(int)
y_pred = (df["verdict"] == "ABNORMAL").astype(int)
y_score = df["abnormal_ratio"]

print(classification_report(y_true, y_pred, target_names=["NORMAL", "ABNORMAL"]))
print(f"ROC AUC: {roc_auc_score(y_true, y_score):.3f}")
print(f"\nConfusion matrix:\n{confusion_matrix(y_true, y_pred)}")

              precision    recall  f1-score   support

      NORMAL       0.67      0.91      0.77       100
    ABNORMAL       0.96      0.82      0.88       247

    accuracy                           0.85       347
   macro avg       0.82      0.87      0.83       347
weighted avg       0.88      0.85      0.85       347

ROC AUC: 0.882

Confusion matrix:
[[ 91   9]
 [ 44 203]]


In [4]:
summary = df.groupby("group").agg(
    total=("file", "count"),
    predicted_abnormal=("verdict", lambda x: (x == "ABNORMAL").sum()),
    accuracy=("verdict", lambda x, gt=df: (x.values == gt.loc[x.index, "label"].values).mean()),
    mean_abnormal_ratio=("abnormal_ratio", "mean"),
    mean_time=("time", "mean"),
).round(3)
summary

,total,predicted_abnormal,accuracy,mean_abnormal_ratio,mean_time
group,,,,,
CT-0,100,9,0.910,0.048,5.908
CT-1,100,68,0.680,0.483,5.854
CT-2,100,90,0.900,0.668,5.816
CT-3,45,43,0.956,0.804,5.533
CT-4,2,2,1.000,1.000,5.100


# MosMedData LDCT with Signs of Lung Cancer (Type I)
- https://mosmed.ai/en/datasets/datasets/mm/

# Dataset of Computed Tomography Angiography (CTA) with Signs of Calcification, Thrombosis, Dilation, and Aneurysm, Including Segmented Abdominal Aorta Lumen and Wall
- https://mosmed.ai/en/datasets/datasets/nabor-dannih-kompyuterno-tomograficheskoi-angiografii-s-priznakami-kaltsinoza-tromboza-dilatatsii-i-anevrizmi-i-soderzhaschii-segmentatsiyu-prosveta-i-stenki-bryushnogo-otdela-aorti/

# RSNA Abdominal Trauma Detection PNG pt1
- https://www.kaggle.com/datasets/theoviel/rsna-abdominal-trauma-detection-png-pt1